# GLUE Statistical Pruning — Full Pipeline

Data-centric statistical pruning on MNLI (generalized z-statistic + cross-task stability check), BERT-base baseline vs. pruned comparison, evaluated on MNLI dev / HANS / SNLI-hard.

In [1]:
!pip install -q -U datasets transformers evaluate accelerate scikit-learn scipy

import os, re, json, time, random, gc
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset as HFDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, matthews_corrcoef, f1_score

WORK_DIR = "/kaggle/working"
os.makedirs(WORK_DIR, exist_ok=True)

MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128
BATCH_SIZE = 64
SEED = 42
EPOCHS = 3
DEVICE = "cuda:0"

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=float)

def load_json(path):
    with open(path) as f:
        return json.load(f)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Setup complete.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 111.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 116.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 59.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 32.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires scipy<1.17,>=1.8, but you have scipy 1.18.1 which is incompatible.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but 

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Setup complete.


## Step 1 — Data acquisition

In [2]:
glue_tasks = ["mnli", "qnli", "rte", "cola", "sst2", "mrpc", "stsb", "qqp", "wnli"]
glue = {}
for task in glue_tasks:
    print(f"Loading GLUE/{task} ...")
    glue[task] = load_dataset("glue", task)

mnli = glue["mnli"]
print("MNLI train/valM/valMM:", len(mnli["train"]), len(mnli["validation_matched"]), len(mnli["validation_mismatched"]))

Loading GLUE/mnli ...


README.md: 0.00B [00:00, ?B/s]

mnli/train-00000-of-00001.parquet:   0%|          | 0.00/52.2M [00:00<?, ?B/s]

mnli/validation_matched-00000-of-00001.p(…):   0%|          | 0.00/1.21M [00:00<?, ?B/s]

mnli/validation_mismatched-00000-of-0000(…):   0%|          | 0.00/1.25M [00:00<?, ?B/s]

...

wnli/test-00000-of-00001.parquet:   0%|          | 0.00/13.6k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/635 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/71 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/146 [00:00<?, ? examples/s]

MNLI train/valM/valMM: 392702 9815 9832


In [3]:
hans_url = "https://raw.githubusercontent.com/tommccoy1/hans/master/heuristics_evaluation_set.txt"
hans_df = pd.read_csv(hans_url, sep="\t")
print("HANS:", hans_df.shape)

import urllib.request
snli_hard_url = "https://nlp.stanford.edu/projects/snli/snli_1.0_test_hard.jsonl"
records = []
with urllib.request.urlopen(snli_hard_url) as f:
    for line in f:
        records.append(json.loads(line))
snli_hard_df = pd.DataFrame(records)
snli_hard_df = snli_hard_df[snli_hard_df["gold_label"] != "-"]
print("SNLI-hard:", snli_hard_df.shape)

HANS: (30000, 11)
SNLI-hard: (3261, 10)


## Step 2 — Feature extraction (Section III-A): unigrams, bigrams, hypothesis length, length ratio, lexical overlap

In [4]:
def tokenize_words(text):
    return re.findall(r"[a-z0-9']+", text.lower())

def get_bigrams(tokens):
    return {f"{tokens[i]}_{tokens[i+1]}" for i in range(len(tokens)-1)} if len(tokens) > 1 else set()

def extract_pair_features(premise, hypothesis):
    p_tokens, h_tokens = tokenize_words(premise), tokenize_words(hypothesis)
    p_set, h_set = set(p_tokens), set(h_tokens)
    hyp_len, prem_len = len(h_tokens), len(p_tokens)
    return {
        "unigrams": list(h_set),
        "bigrams": list(get_bigrams(h_tokens)),
        "hyp_len": hyp_len,
        "prem_len": prem_len,
        "length_ratio": hyp_len / prem_len if prem_len > 0 else 0.0,
        "lexical_overlap": len(p_set & h_set) / hyp_len if hyp_len > 0 else 0.0,
    }

def add_pair_features_batch(batch, premise_col, hypothesis_col):
    out = {"unigrams": [], "bigrams": [], "hyp_len": [], "prem_len": [], "length_ratio": [], "lexical_overlap": []}
    for p, h in zip(batch[premise_col], batch[hypothesis_col]):
        f = extract_pair_features(p, h)
        for k in out:
            out[k].append(f[k])
    return out

mnli_train_feats = mnli["train"].map(lambda b: add_pair_features_batch(b, "premise", "hypothesis"), batched=True, batch_size=1000)
qnli_train_feats = glue["qnli"]["train"].map(lambda b: add_pair_features_batch(b, "sentence", "question"), batched=True, batch_size=1000)
rte_train_feats  = glue["rte"]["train"].map(lambda b: add_pair_features_batch(b, "sentence1", "sentence2"), batched=True, batch_size=1000)

print("Feature extraction complete.")
print("MNLI avg lexical overlap:", np.mean(mnli_train_feats["lexical_overlap"]))

Map:   0%|          | 0/392702 [00:00<?, ? examples/s]

Map:   0%|          | 0/104743 [00:00<?, ? examples/s]

Map:   0%|          | 0/2490 [00:00<?, ? examples/s]

Feature extraction complete.
MNLI avg lexical overlap: 0.46127246702308417


## Step 3 — Generalized z-statistic (Eq. 2, Section IV-A)

In [5]:
from collections import defaultdict

def compute_class_prior(labels, num_classes=None):
    labels = np.array(labels)
    if num_classes is None:
        num_classes = len(np.unique(labels))
    counts = np.bincount(labels, minlength=num_classes)
    return counts / counts.sum()

def compute_zstat_for_features(feature_lists, labels, p0, min_count=20):
    feat_label_counts = defaultdict(lambda: np.zeros(len(p0), dtype=np.int64))
    feat_counts = defaultdict(int)
    for feats, l in zip(feature_lists, labels):
        for f in set(feats):
            feat_label_counts[f][l] += 1
            feat_counts[f] += 1
    results = {}
    for f, n in feat_counts.items():
        if n < min_count:
            continue
        counts = feat_label_counts[f]
        p_hat = counts / n
        z = (p_hat - p0) / np.sqrt(p0 * (1 - p0) / n)
        best_label = int(np.argmax(z))
        results[f] = {"n": int(n), "best_label": best_label, "max_z": float(z[best_label])}
    return results

def bucket_continuous(values, names=("low", "med", "high")):
    values = np.array(values)
    edges = np.quantile(values, [0, 1/3, 2/3, 1.0])
    edges[-1] += 1e-9
    idx = np.digitize(values, edges[1:-1], right=False)
    return [names[i] for i in idx]

def make_feature_list(buckets, prefix):
    return [[f"{prefix}_{b}"] for b in buckets]

def compute_surface_z(feats_dict, labels, p0, min_count=1):
    surface_feats = {
        "hyplen":   make_feature_list(bucket_continuous(feats_dict["hyp_len"]), "hyplen"),
        "lenratio": make_feature_list(bucket_continuous(feats_dict["length_ratio"]), "lenratio"),
        "overlap":  make_feature_list(bucket_continuous(feats_dict["lexical_overlap"]), "overlap"),
    }
    z = {}
    for name, feats in surface_feats.items():
        z.update(compute_zstat_for_features(feats, labels, p0, min_count=min_count))
    return z

mnli_labels = mnli["train"]["label"]
mnli_p0 = compute_class_prior(mnli_labels, num_classes=3)
mnli_unigram_z = compute_zstat_for_features(mnli_train_feats["unigrams"], mnli_labels, mnli_p0, min_count=20)
mnli_bigram_z  = compute_zstat_for_features(mnli_train_feats["bigrams"],  mnli_labels, mnli_p0, min_count=10)
mnli_surface_z = compute_surface_z(mnli_train_feats, mnli_labels, mnli_p0, min_count=1)

qnli_labels = glue["qnli"]["train"]["label"]
qnli_p0 = compute_class_prior(qnli_labels, num_classes=2)
qnli_unigram_z = compute_zstat_for_features(qnli_train_feats["unigrams"], qnli_labels, qnli_p0, min_count=20)
qnli_bigram_z  = compute_zstat_for_features(qnli_train_feats["bigrams"],  qnli_labels, qnli_p0, min_count=10)
qnli_surface_z = compute_surface_z(qnli_train_feats, qnli_labels, qnli_p0, min_count=1)

rte_labels = glue["rte"]["train"]["label"]
rte_p0 = compute_class_prior(rte_labels, num_classes=2)
rte_unigram_z = compute_zstat_for_features(rte_train_feats["unigrams"], rte_labels, rte_p0, min_count=5)
rte_bigram_z  = compute_zstat_for_features(rte_train_feats["bigrams"],  rte_labels, rte_p0, min_count=5)
rte_surface_z = compute_surface_z(rte_train_feats, rte_labels, rte_p0, min_count=1)

print("MNLI class prior:", mnli_p0)
print("QNLI class prior:", qnli_p0)
print("RTE class prior:", rte_p0)
print("Top MNLI unigram:", max(mnli_unigram_z.items(), key=lambda kv: kv[1]["max_z"]))

MNLI class prior: [0.33332909 0.33333164 0.33333928]
QNLI class prior: [0.50005251 0.49994749]
RTE class prior: [0.50160643 0.49839357]
Top MNLI unigram: ('no', {'n': 16369, 'best_label': 2, 'max_z': 114.81166625346975})


## Step 4 — Cross-task stability check (Ambiguity Detection, Section IV-B)

In [6]:
def label_family(task_name, best_label):
    return "entailment" if best_label == 0 else "non_entailment"

def check_stability(feature, per_task_z, z_threshold=5.0, min_tasks=2):
    hits = []
    for task, z_dict in per_task_z.items():
        info = z_dict.get(feature)
        if info is not None and info["max_z"] >= z_threshold:
            hits.append((task, label_family(task, info["best_label"])))
    if len(hits) < min_tasks:
        return False, hits
    families = {fam for _, fam in hits}
    return len(families) == 1, hits

per_task_unigram_z = {"mnli": mnli_unigram_z, "qnli": qnli_unigram_z, "rte": rte_unigram_z}
per_task_bigram_z  = {"mnli": mnli_bigram_z,  "qnli": qnli_bigram_z,  "rte": rte_bigram_z}
per_task_surface_z = {"mnli": mnli_surface_z, "qnli": qnli_surface_z, "rte": rte_surface_z}

stability_results = {}
for feat_type, per_task in [("unigram", per_task_unigram_z), ("bigram", per_task_bigram_z), ("surface", per_task_surface_z)]:
    all_feats = set()
    for d in per_task.values():
        all_feats |= set(d.keys())
    stable_feats = [feat for feat in all_feats if check_stability(feat, per_task)[0]]
    stability_results[feat_type] = stable_feats
    print(f"{feat_type}: {len(stable_feats)} stable features out of {len(all_feats)} candidates")

save_json(stability_results, f"{WORK_DIR}/stability_results.json")
print("Stable surface features:", stability_results["surface"])

unigram: 0 stable features out of 11569 candidates
bigram: 0 stable features out of 48494 candidates
surface: 3 stable features out of 9 candidates
Stable surface features: ['overlap_med', 'overlap_low', 'overlap_high']


## Step 5 — Statistical pruning filter (Section III-A / Fig. 1)

Prunes every surface-feature bucket the Step 4 stability check actually confirmed as stable (not just the two most extreme ones) — for each stable bucket, remove just enough of its over-represented label to bring that bucket's conditional distribution back to the task's empirical prior, ranked by how extreme each example is within the bucket.

In [7]:
FEATURE_KEY_MAP = {"hyplen": "hyp_len", "lenratio": "length_ratio", "overlap": "lexical_overlap"}

def compute_removal_count(label_count, bucket_n, p0_label):
    r = (label_count - p0_label * bucket_n) / (1 - p0_label)
    return max(0, int(round(r)))

labels_arr = np.array(mnli_labels)
remove_indices = set()
pruning_log = {}

for feat in stability_results["surface"]:
    prefix, bucket_name = feat.rsplit("_", 1)
    key = FEATURE_KEY_MAP[prefix]
    values = np.array(mnli_train_feats[key])
    buckets = np.array(bucket_continuous(values))
    mask = buckets == bucket_name
    n = int(mask.sum())
    counts = np.bincount(labels_arr[mask], minlength=3)
    p_hat = counts / n
    over_label = int(np.argmax(p_hat - mnli_p0))
    if p_hat[over_label] <= mnli_p0[over_label]:
        pruning_log[feat] = {"n": n, "removed": 0, "note": "not over-represented, skipped"}
        continue

    r = compute_removal_count(int(counts[over_label]), n, mnli_p0[over_label])
    idx_pool = np.where(mask & (labels_arr == over_label))[0]
    scores = values[idx_pool]
    if bucket_name == "high":
        order = np.argsort(-scores)          # most extreme (highest) first
    elif bucket_name == "low":
        order = np.argsort(scores)           # most extreme (lowest) first
    else:  # "med" -- no natural extremity direction; use a fixed, reproducible ordering
        order = np.random.RandomState(SEED).permutation(len(idx_pool))
    removed_this_feat = set(idx_pool[order][:r].tolist())
    remove_indices |= removed_this_feat
    pruning_log[feat] = {"n": n, "over_label": int(over_label), "removed": len(removed_this_feat)}
    print(f"{feat:15s} n={n:6d}  over-represented label={over_label} ({p_hat[over_label]:.3f} vs prior {mnli_p0[over_label]:.3f})  removing={len(removed_this_feat)}")

keep_mask = np.ones(len(mnli["train"]), dtype=bool)
keep_mask[list(remove_indices)] = False

mnli_train_pruned = mnli["train"].select(np.where(keep_mask)[0])
mnli_train_pruned.to_parquet(f"{WORK_DIR}/pruned_mnli_train.parquet")

pruned_labels = np.array(mnli_train_pruned["label"])
pruned_counts = np.bincount(pruned_labels, minlength=3)

table_ii = {
    "original": {"train": len(mnli["train"]), "valid_m": 9815, "valid_mm": 9832, "total": 412349},
    "pruned": {"train": len(mnli_train_pruned), "valid_m": 9815, "valid_mm": 9832,
               "total": len(mnli_train_pruned) + 9815 + 9832},
    "pruned_class_balance": {"entailment": float(pruned_counts[0]/len(pruned_labels)),
                              "neutral": float(pruned_counts[1]/len(pruned_labels)),
                              "contradiction": float(pruned_counts[2]/len(pruned_labels))},
    "pruning_log": pruning_log,
}
save_json(table_ii, f"{WORK_DIR}/table_ii.json")
print()
print(json.dumps(table_ii, indent=2))

overlap_med     n=125051  over-represented label=1 (0.362 vs prior 0.333)  removing=5390
overlap_low     n=130673  over-represented label=1 (0.475 vs prior 0.333)  removing=27797
overlap_high    n=136978  over-represented label=0 (0.541 vs prior 0.333)  removing=42590


Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]


{
  "original": {
    "train": 392702,
    "valid_m": 9815,
    "valid_mm": 9832,
    "total": 412349
  },
  "pruned": {
    "train": 316925,
    "valid_m": 9815,
    "valid_mm": 9832,
    "total": 336572
  },
  "pruned_class_balance": {
    "entailment": 0.27864321211643134,
    "neutral": 0.30831584759801217,
    "contradiction": 0.41304094028555655
  },
  "pruning_log": {
    "overlap_med": {
      "n": 125051,
      "over_label": 1,
      "removed": 5390
    },
    "overlap_low": {
      "n": 130673,
      "over_label": 1,
      "removed": 27797
    },
    "overlap_high": {
      "n": 136978,
      "over_label": 0,
      "removed": 42590
    }
  }
}


## Step 6 — Tokenization (dynamic padding)

In [8]:
def tokenize_and_prepare_dynamic(dataset, text1="premise", text2="hypothesis", max_length=MAX_LEN):
    def tok_fn(batch):
        return tokenizer(batch[text1], batch[text2], truncation=True, max_length=max_length)
    ds = dataset.map(tok_fn, batched=True, remove_columns=[c for c in dataset.column_names if c not in ("label",)])
    ds = ds.rename_column("label", "labels")
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "token_type_ids", "labels"])
    return ds

train_original_tok = tokenize_and_prepare_dynamic(mnli["train"])
train_pruned_tok = tokenize_and_prepare_dynamic(mnli_train_pruned)
val_matched_tok = tokenize_and_prepare_dynamic(mnli["validation_matched"])
val_mismatched_tok = tokenize_and_prepare_dynamic(mnli["validation_mismatched"])

def build_eval_loader(df, s1_col, s2_col, label_col, label_map, batch_size=128):
    df = df.copy()
    df["label_int"] = df[label_col].map(label_map)
    ds = HFDataset.from_pandas(df[[s1_col, s2_col, "label_int"]].rename(
        columns={s1_col: "text1", s2_col: "text2", "label_int": "labels"}))
    def tok_fn(batch):
        return tokenizer(batch["text1"], batch["text2"], truncation=True, max_length=MAX_LEN)
    ds = ds.map(tok_fn, batched=True, remove_columns=["text1", "text2"])
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "token_type_ids", "labels"])
    return DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=data_collator)

hans_label_map = {"entailment": 0, "non-entailment": 1}
snli_hard_label_map = {"entailment": 0, "neutral": 1, "contradiction": 2}
hans_loader = build_eval_loader(hans_df, "sentence1", "sentence2", "gold_label", hans_label_map)
snli_hard_loader = build_eval_loader(snli_hard_df, "sentence1", "sentence2", "gold_label", snli_hard_label_map)

train_loader_original = DataLoader(train_original_tok, batch_size=BATCH_SIZE, shuffle=True, collate_fn=data_collator)
train_loader_pruned = DataLoader(train_pruned_tok, batch_size=BATCH_SIZE, shuffle=True, collate_fn=data_collator)
val_matched_loader = DataLoader(val_matched_tok, batch_size=128, shuffle=False, collate_fn=data_collator)
val_mismatched_loader = DataLoader(val_mismatched_tok, batch_size=128, shuffle=False, collate_fn=data_collator)

print("Tokenization complete.")
print("Train batches: original", len(train_loader_original), "pruned", len(train_loader_pruned))

Map:   0%|          | 0/392702 [00:00<?, ? examples/s]

Map:   0%|          | 0/316925 [00:00<?, ? examples/s]

Map:   0%|          | 0/9815 [00:00<?, ? examples/s]

Map:   0%|          | 0/9832 [00:00<?, ? examples/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3261 [00:00<?, ? examples/s]

Tokenization complete.
Train batches: original 6136 pruned 4952


## Step 7 — Training (BERT-base, manual loop on single GPU — avoids the DataParallel slowdown seen with `Trainer` on multi-GPU)

Two additions vs. the interactive run: a linear warmup+decay LR schedule with gradient clipping (the standard BERT fine-tuning recipe — we'd been running with a flat LR before), and per-epoch checkpointing with model selection by the model's *own* MNLI validation accuracy (early stopping), rather than always using the final epoch. This directly addresses the overfitting we diagnosed in the pruned condition (epoch 3 was worse than epoch 2 on MNLI dev).

In [9]:
def train_epoch(model, loader, optimizer, scaler, scheduler, device=DEVICE):
    model.train()
    total_loss = 0.0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            out = model(**batch)
        scaler.scale(out.loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += out.loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, device=DEVICE):
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            out = model(**{k: v for k, v in batch.items() if k != "labels"})
        preds = out.logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(batch["labels"].cpu().numpy())
    return {"accuracy": accuracy_score(all_labels, all_preds),
            "mcc": matthews_corrcoef(all_labels, all_preds),
            "f1": f1_score(all_labels, all_preds, average="macro")}

@torch.no_grad()
def get_predictions_3way(model, loader, device=DEVICE):
    model.eval()
    all_preds = []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            out = model(**{k: v for k, v in batch.items() if k != "labels"})
        all_preds.extend(out.logits.argmax(dim=-1).cpu().numpy())
    return np.array(all_preds)

def evaluate_hans(model, loader, device=DEVICE):
    preds_3way = get_predictions_3way(model, loader, device)
    preds_binary = np.where(preds_3way == 0, 0, 1)
    all_labels = []
    for batch in loader:
        all_labels.extend(batch["labels"].numpy())
    all_labels = np.array(all_labels)
    return {"accuracy": accuracy_score(all_labels, preds_binary),
            "mcc": matthews_corrcoef(all_labels, preds_binary),
            "f1": f1_score(all_labels, preds_binary, average="macro")}

print("Train/eval functions ready.")

Train/eval functions ready.


In [30]:
def run_full_training(train_loader, condition_name, seed=SEED, epochs=EPOCHS):
    best_ckpt_path = f"{WORK_DIR}/{condition_name}_seed{seed}_best"
    history_path = f"{WORK_DIR}/{condition_name}_seed{seed}_history.json"

    if os.path.exists(os.path.join(best_ckpt_path, "config.json")) and os.path.exists(history_path):
        history = load_json(history_path)
        if len(history) >= epochs:
            print(f"[{condition_name}] complete {epochs}-epoch checkpoint already exists on disk -- loading instead of retraining.")
            model = AutoModelForSequenceClassification.from_pretrained(best_ckpt_path).to(DEVICE)
            return model, history
        else:
            print(f"[{condition_name}] found an INCOMPLETE history ({len(history)}/{epochs} epochs, likely from an "
                  f"interrupted run) -- optimizer/scheduler state wasn't saved so resuming mid-schedule isn't safe. "
                  f"Retraining this condition from scratch to guarantee a fair {epochs}-epoch comparison.")

    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    scaler = torch.cuda.amp.GradScaler()
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
    )

    history = []
    best_avg_acc = -1.0
    for epoch in range(epochs):
        t0 = time.time()
        loss = train_epoch(model, train_loader, optimizer, scaler, scheduler)
        val_m = evaluate(model, val_matched_loader)
        val_mm = evaluate(model, val_mismatched_loader)
        avg_acc = (val_m["accuracy"] + val_mm["accuracy"]) / 2
        dt = time.time() - t0
        print(f"[{condition_name}] seed={seed} epoch={epoch+1}/{epochs} loss={loss:.4f} "
              f"valM={val_m} valMM={val_mm} avg_acc={avg_acc:.4f}  ({dt/60:.1f} min)")
        history.append({"epoch": epoch+1, "loss": loss, "val_matched": val_m, "val_mismatched": val_mm, "avg_acc": avg_acc})
        epoch_ckpt_path = f"{WORK_DIR}/{condition_name}_seed{seed}_epoch{epoch+1}"
        model.save_pretrained(epoch_ckpt_path)
        save_json(history, history_path)
        if avg_acc > best_avg_acc:
            best_avg_acc = avg_acc
            model.save_pretrained(best_ckpt_path)
            print(f"  -> new best checkpoint (avg MNLI dev acc={avg_acc:.4f})")

    model = AutoModelForSequenceClassification.from_pretrained(best_ckpt_path).to(DEVICE)
    print(f"[{condition_name}] done. Best epoch avg dev acc={best_avg_acc:.4f}, loaded from {best_ckpt_path}")
    return model, history

print("Patched run_full_training ready.")

Patched run_full_training ready.


In [11]:
print("=== ORIGINAL condition (~90-100 min if not already checkpointed) ===")
model_original, history_original = run_full_training(train_loader_original, "original", seed=SEED)
del model_original
gc.collect()
torch.cuda.empty_cache()
print("Original condition done; best-epoch model freed from memory (still saved on disk).")

=== ORIGINAL condition (~90-100 min if not already checkpointed) ===


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_58/3533

[original] seed=42 epoch=1/3 loss=0.5837 valM={'accuracy': 0.8212939378502292, 'mcc': 0.7332085317281453, 'f1': 0.8198130180513908} valMM={'accuracy': 0.8294344995931652, 'mcc': 0.7446336774644787, 'f1': 0.8276248750121084} avg_acc=0.8254  (31.7 min)


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> new best checkpoint (avg MNLI dev acc=0.8441)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[original] done. Best epoch avg dev acc=0.8441, loaded from /kaggle/working/original_seed42_best
Original condition done; best-epoch model freed from memory (still saved on disk).


In [31]:
print("=== PRUNED condition (retraining, ~75-85 min) ===")
model_pruned, history_pruned = run_full_training(train_loader_pruned, "pruned", seed=SEED)
del model_pruned
gc.collect()
torch.cuda.empty_cache()
print("Pruned condition done; best-epoch model freed from memory (still saved on disk).")

=== PRUNED condition (retraining, ~75-85 min) ===
[pruned] found an INCOMPLETE history (2/3 epochs, likely from an interrupted run) -- optimizer/scheduler state wasn't saved so resuming mid-schedule isn't safe. Retraining this condition from scratch to guarantee a fair 3-epoch comparison.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_58/6200

[pruned] seed=42 epoch=1/3 loss=0.6204 valM={'accuracy': 0.7968415690269995, 'mcc': 0.6978113294510015, 'f1': 0.7942949875626745} valMM={'accuracy': 0.7963791700569569, 'mcc': 0.6965278275435762, 'f1': 0.7938116016453617} avg_acc=0.7966  (25.9 min)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> new best checkpoint (avg MNLI dev acc=0.8077)
[pruned] seed=42 epoch=3/3 loss=0.2920 valM={'accuracy': 0.7997962302598064, 'mcc': 0.7026004390977503, 'f1': 0.799030143873184} valMM={'accuracy': 0.8026851098454028, 'mcc': 0.7060609552211563, 'f1': 0.8019377136530578} avg_acc=0.8012  (25.8 min)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[pruned] done. Best epoch avg dev acc=0.8077, loaded from /kaggle/working/pruned_seed42_best
Pruned condition done; best-epoch model freed from memory (still saved on disk).


## Step 8 — Evaluation on MNLI dev, HANS, SNLI-hard

In [32]:
model_original = AutoModelForSequenceClassification.from_pretrained(f"{WORK_DIR}/original_seed{SEED}_best").to(DEVICE)
model_pruned = AutoModelForSequenceClassification.from_pretrained(f"{WORK_DIR}/pruned_seed{SEED}_best").to(DEVICE)

hans_original = evaluate_hans(model_original, hans_loader)
snli_hard_original = evaluate(model_original, snli_hard_loader)
hans_pruned = evaluate_hans(model_pruned, hans_loader)
snli_hard_pruned = evaluate(model_pruned, snli_hard_loader)

print("=== ORIGINAL ===")
print("HANS:", hans_original)
print("SNLI-hard:", snli_hard_original)
print("=== PRUNED ===")
print("HANS:", hans_pruned)
print("SNLI-hard:", snli_hard_pruned)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

=== ORIGINAL ===
HANS: {'accuracy': 0.5281, 'mcc': 0.14165638839742348, 'f1': 0.40216603560163966}
SNLI-hard: {'accuracy': 0.7163446795461514, 'mcc': 0.5752520121738652, 'f1': 0.7122857645366077}
=== PRUNED ===
HANS: {'accuracy': 0.5745, 'mcc': 0.15628057137538728, 'f1': 0.5645942160500099}
SNLI-hard: {'accuracy': 0.6657467034651947, 'mcc': 0.5014643585774802, 'f1': 0.6584865391310135}


## Step 9 — HANS breakdown by heuristic, and by heuristic x gold label

In [33]:
def get_hans_predictions(model, loader, device=DEVICE):
    preds_3way = get_predictions_3way(model, loader, device)
    return np.where(preds_3way == 0, "entailment", "non-entailment")

hans_df["pred_original"] = get_hans_predictions(model_original, hans_loader)
hans_df["pred_pruned"] = get_hans_predictions(model_pruned, hans_loader)

print("=== By heuristic ===")
for heuristic in hans_df["heuristic"].unique():
    sub = hans_df[hans_df["heuristic"] == heuristic]
    acc_orig = (sub["pred_original"] == sub["gold_label"]).mean()
    acc_pruned = (sub["pred_pruned"] == sub["gold_label"]).mean()
    print(f"{heuristic:20s} orig={acc_orig:.3f}  pruned={acc_pruned:.3f}  delta={acc_pruned-acc_orig:+.3f}")

print()
print("=== By heuristic x gold label ===")
for heuristic in hans_df["heuristic"].unique():
    for gold in ["entailment", "non-entailment"]:
        sub = hans_df[(hans_df["heuristic"] == heuristic) & (hans_df["gold_label"] == gold)]
        acc_orig = (sub["pred_original"] == sub["gold_label"]).mean()
        acc_pruned = (sub["pred_pruned"] == sub["gold_label"]).mean()
        print(f"{heuristic:20s} {gold:15s} n={len(sub):5d}  orig={acc_orig:.3f}  pruned={acc_pruned:.3f}  delta={acc_pruned-acc_orig:+.3f}")

hans_df.to_csv(f"{WORK_DIR}/hans_predictions.csv", index=False)

=== By heuristic ===
lexical_overlap      orig=0.531  pruned=0.501  delta=-0.031
subsequence          orig=0.514  pruned=0.611  delta=+0.097
constituent          orig=0.539  pruned=0.612  delta=+0.073

=== By heuristic x gold label ===
lexical_overlap      entailment      n= 5000  orig=0.974  pruned=0.374  delta=-0.600
lexical_overlap      non-entailment  n= 5000  orig=0.089  pruned=0.628  delta=+0.539
subsequence          entailment      n= 5000  orig=0.997  pruned=0.415  delta=-0.582
subsequence          non-entailment  n= 5000  orig=0.031  pruned=0.807  delta=+0.776
constituent          entailment      n= 5000  orig=0.990  pruned=0.482  delta=-0.508
constituent          non-entailment  n= 5000  orig=0.088  pruned=0.742  delta=+0.654


## Step 10 — Final results summary (saved to results_summary.json)

In [40]:
results_summary = {
    "table_ii": table_ii,
    "mnli_val_final_epoch": {"original": history_original[-1], "pruned": history_pruned[-1]},
    "hans_topline": {"original": hans_original, "pruned": hans_pruned},
    "snli_hard": {"original": snli_hard_original, "pruned": snli_hard_pruned},
}
save_json(results_summary, f"{WORK_DIR}/results_summary.json")

print("=" * 70)
print("FINAL RESULTS SUMMARY")
print("=" * 70)
print(json.dumps(results_summary, indent=2))
print()
print("All results saved to:", WORK_DIR)
print("Files:", sorted(os.listdir(WORK_DIR)))

FINAL RESULTS SUMMARY
{
  "table_ii": {
    "original": {
      "train": 392702,
      "valid_m": 9815,
      "valid_mm": 9832,
      "total": 412349
    },
    "pruned": {
      "train": 316925,
      "valid_m": 9815,
      "valid_mm": 9832,
      "total": 336572
    },
    "pruned_class_balance": {
      "entailment": 0.27864321211643134,
      "neutral": 0.30831584759801217,
      "contradiction": 0.41304094028555655
    },
    "pruning_log": {
      "overlap_med": {
        "n": 125051,
        "over_label": 1,
        "removed": 5390
      },
      "overlap_low": {
        "n": 130673,
        "over_label": 1,
        "removed": 27797
      },
      "overlap_high": {
        "n": 136978,
        "over_label": 0,
        "removed": 42590
      }
    }
  },
  "mnli_val_final_epoch": {
    "original": {
      "epoch": 3,
      "loss": 0.27744278176341597,
      "val_matched": {
        "accuracy": 0.8435048395313296,
        "mcc": 0.765537419532012,
        "f1": 0.8429326515136198


## Step 11 — MNLI metrics table and confusion matrices (Table VI, Fig. 5)Reuses `model_original`, `model_pruned`, and the loaders already built in Steps 6–8 — no new downloads or retraining, just a closer look at what the two checkpoints actually got right and wrong on MNLI-matched and mismatched dev.

In [ ]:
from sklearn.metrics import precision_score, recall_score, confusion_matriximport pandas as pdLABEL_NAMES = ["entailment", "neutral", "contradiction"]@torch.no_grad()def evaluate_full(model, loader, device=DEVICE):    model.eval()    all_preds, all_labels = [], []    for batch in loader:        batch = {k: v.to(device) for k, v in batch.items()}        with torch.autocast(device_type="cuda", dtype=torch.float16):            out = model(**{k: v for k, v in batch.items() if k != "labels"})        all_preds.extend(out.logits.argmax(dim=-1).cpu().numpy())        all_labels.extend(batch["labels"].cpu().numpy())    all_preds, all_labels = np.array(all_preds), np.array(all_labels)    metrics = {        "accuracy":  accuracy_score(all_labels, all_preds),        "precision": precision_score(all_labels, all_preds, average="macro"),        "recall":    recall_score(all_labels, all_preds, average="macro"),        "f1":        f1_score(all_labels, all_preds, average="macro"),    }    return metrics, all_labels, all_predsrows = {}preds_cache = {}for name, model in [("Original", model_original), ("Pruned", model_pruned)]:    for split_name, loader in [("Matched", val_matched_loader), ("Mismatched", val_mismatched_loader)]:        m, gold, pred = evaluate_full(model, loader)        rows[f"{name} ({split_name})"] = m        preds_cache[(name, split_name)] = (gold, pred)metrics_df = pd.DataFrame(rows).T.round(4)print(metrics_df)metrics_df.to_csv(f"{WORK_DIR}/mnli_full_metrics_table.csv")gold_o, pred_o = preds_cache[("Original", "Matched")]gold_p, pred_p = preds_cache[("Pruned", "Matched")]cm_orig   = confusion_matrix(gold_o, pred_o, labels=[0, 1, 2])cm_pruned = confusion_matrix(gold_p, pred_p, labels=[0, 1, 2])print("\nConfusion matrix (original):\n", cm_orig)print("Confusion matrix (pruned):\n", cm_pruned)short = ["entail.", "neutral", "contra."]fig, axes = plt.subplots(1, 2, figsize=(6.6, 3.0))for ax, cm, title in zip(axes, [cm_orig, cm_pruned], ["Original", "Pruned"]):    ax.imshow(cm, cmap="Blues", vmin=0, vmax=3000)    ax.set_xticks(range(3)); ax.set_xticklabels(short, fontsize=8.5)    ax.set_yticks(range(3)); ax.set_yticklabels(short, fontsize=8.5)    ax.set_xlabel("Predicted", fontsize=9)    if title == "Original":        ax.set_ylabel("Gold", fontsize=9)    ax.set_title(f"{title} (MNLI-matched dev)", fontsize=9.5)    for i in range(3):        for j in range(3):            ax.text(j, i, f"{cm[i,j]:,}", ha="center", va="center", fontsize=9.5,                     color="white" if cm[i, j] > 1500 else "black")plt.tight_layout(pad=0.4)plt.savefig(f"{WORK_DIR}/mnli_confusion_matrices.pdf", bbox_inches="tight")plt.savefig(f"{WORK_DIR}/mnli_confusion_matrices.png", dpi=300, bbox_inches="tight")plt.show()

## Step 12 — HANS confusion matrix and qualitative examples (Fig. 4, Section VI-B)`hans_df` already has `pred_original` and `pred_pruned` columns from Step 9 — this just turns those predictions into the confusion matrices and the corrected/overshoot examples quoted in the paper.

In [ ]:
short_bin = ["entail.", "non-entail."]fig, axes = plt.subplots(1, 2, figsize=(6.6, 3.0))for ax, cond, title in zip(axes, ["original", "pruned"], ["Original", "Pruned"]):    cm = confusion_matrix(hans_df["gold_label"], hans_df[f"pred_{cond}"], labels=["entailment", "non-entailment"])    ax.imshow(cm, cmap="Blues", vmin=0, vmax=15000)    ax.set_xticks([0, 1]); ax.set_xticklabels(short_bin, fontsize=8.5)    ax.set_yticks([0, 1]); ax.set_yticklabels(short_bin, fontsize=8.5)    ax.set_xlabel("Predicted", fontsize=9)    if cond == "original":        ax.set_ylabel("Gold", fontsize=9)    ax.set_title(f"{title} (HANS, n=30,000)", fontsize=9.5)    for i in range(2):        for j in range(2):            ax.text(j, i, f"{cm[i,j]:,}", ha="center", va="center", fontsize=10.5,                     color="white" if cm[i, j] > 7500 else "black")plt.tight_layout(pad=0.4)plt.savefig(f"{WORK_DIR}/hans_confusion_matrices.pdf", bbox_inches="tight")plt.savefig(f"{WORK_DIR}/hans_confusion_matrices.png", dpi=300, bbox_inches="tight")plt.show()p_o = precision_score(hans_df["gold_label"], hans_df["pred_original"], average="macro")r_o = recall_score(hans_df["gold_label"], hans_df["pred_original"], average="macro")p_p = precision_score(hans_df["gold_label"], hans_df["pred_pruned"], average="macro")r_p = recall_score(hans_df["gold_label"], hans_df["pred_pruned"], average="macro")print(f"Original: precision={p_o:.4f} recall={r_o:.4f}")print(f"Pruned:   precision={p_p:.4f} recall={r_p:.4f}")lex = hans_df[hans_df["heuristic"] == "lexical_overlap"].reset_index(drop=True)corrected = lex[(lex["pred_original"] != lex["gold_label"]) & (lex["pred_pruned"] == lex["gold_label"])]overshoot_ent = lex[(lex["gold_label"] == "entailment") &                     (lex["pred_original"] == "entailment") &                     (lex["pred_pruned"] == "non-entailment")]print(f"\nCorrected by pruning: {len(corrected)} / {len(lex)} lexical-overlap pairs")print(f"Overshoot on genuine entailment pairs: {len(overshoot_ent)} / "      f"{len(lex[lex['gold_label']=='entailment'])} entailment-labeled pairs")print("\n=== Representative corrected example ===")row = corrected.iloc[0]print(f"Premise: {row['sentence1']}\nHypothesis: {row['sentence2']}\nGold: {row['gold_label']}")print("\n=== Representative overshoot example ===")row = overshoot_ent.iloc[0]print(f"Premise: {row['sentence1']}\nHypothesis: {row['sentence2']}\nGold: entailment")